# **MNIST DIGIT CLASSIFIER USING CNN**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Loading the dataset

In [4]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 342kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


# Dataloader

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True
)
test_loader = DataLoader(
    test_dataset,
    batch_size = 64,
    shuffle = False
)

# Augmentation

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)), 
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# Model

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

In [11]:
model = CNN().to(device)
print("Model instantiated and moved to device.")

Model instantiated and moved to device.


# Loss Function

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)

# Training

In [17]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')


Epoch 1/10, Loss: 0.0019
Epoch 2/10, Loss: 0.0098
Epoch 3/10, Loss: 0.1107
Epoch 4/10, Loss: 0.0042
Epoch 5/10, Loss: 0.0801
Epoch 6/10, Loss: 0.0110
Epoch 7/10, Loss: 0.0045
Epoch 8/10, Loss: 0.0138
Epoch 9/10, Loss: 0.0175
Epoch 10/10, Loss: 0.0005


# Evaluation

In [18]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output = model(data)
    _, predicted = torch.max(output.data, 1)
    total+=target.size(0)
    correct+=(predicted == target).sum().item()
print(f'Accuracy: {100*correct/total}%')

Accuracy: 99.21%
